# 02 表格价值学习：用下一步修正当前估计

这一课只回答一个问题：**不等一个回合结束，怎样利用刚刚看到的一步转移，修正当前状态的价值估计？**

上一课的 GridWorld 给出了状态、奖励和折扣回报。本课先固定一条策略，不学习选动作；我们只学习怎样评价这条策略。TD（Temporal-Difference，时序差分）正是这种“边走边更新”的办法。

## 本课目标

读完后，你应能：

1. 说出状态价值 $V^{\pi}(s)$ 在评价什么；
2. 写出一次 TD(0) 更新，并指出其中的目标和误差；
3. 用纸笔完成一个两状态的更新；
4. 解释 TD 与蒙特卡洛方法在“何时更新”上的区别。

本课不训练智能体、不比较策略，也不引入 Q-learning。先把“估计如何被数据纠正”这件事弄清楚。

## 1. 价值是在评价一条固定策略

设策略为 $\pi$。从状态 $s$ 出发，始终按 $\pi$ 行动，未来折扣回报的平均值称为**状态价值**：

$$
V^{\pi}(s)=\mathbb{E}_{\pi}[G_t\mid S_t=s]
$$

它回答的不是“此刻该选哪个动作”，而是“若以后一直按这条策略走，这个状态大致有多好”。

例如，若一条固定策略总是从某格走向出口，那么靠近出口的格子通常价值更高；通往陷阱的格子通常价值更低。开始时，我们不知道这些数，只能用一张表保存猜测值 $V(s)$，再由经验逐步修正。

## 2. 为什么不总等到回合结束？

蒙特卡洛方法会走完整个回合，得到真实经历的回报 $G_t$ 后，再把 $V(S_t)$ 朝 $G_t$ 移动。这很直观，但有两个限制：

- 必须等到终点，长回合中的早期状态要等很久；
- 如果任务没有自然终点，例如持续控制温度，就没有一个完整回合可等。

TD 的想法是：刚观察到 $S_t\rightarrow S_{t+1}$ 和奖励 $R_{t+1}$ 时，就让下一状态的当前估计替我们补上更远的未来。

## 3. 一步 TD 目标

一次经验包含当前状态 $S_t$、一步奖励 $R_{t+1}$ 和下一状态 $S_{t+1}$。TD 不使用完整回报，而使用下面的一步目标：

$$
\text{TD target}=R_{t+1}+\gamma V(S_{t+1})
$$

其中 $R_{t+1}$ 是已经观察到的事实；$V(S_{t+1})$ 是目前对更远未来的估计。把一个估计放进目标来更新另一个估计，叫作**自举**（bootstrapping）。

这里的 $V$ 都是更新前表中的数值。一次转移只直接修改 $S_t$ 的表项，不要误以为这一步也会同时重算整张表。

## 4. 从目标到更新公式

当前估计与目标的差叫作 TD 误差：

$$
\delta_t=R_{t+1}+\gamma V(S_{t+1})-V(S_t)
$$

用步长 $\alpha$ 控制每次改动的幅度，TD(0) 更新为：

$$
V(S_t)\leftarrow V(S_t)+\alpha\delta_t
$$

若 $\delta_t>0$，说明刚看到的结果比原先预期好，价值上调；若 $\delta_t<0$，说明原先过于乐观，价值下调。$\alpha$ 较大时学得快但单次经验影响更大；较小时更平稳但改得更慢。

## 5. 手算一个最小例子

某策略下，智能体从状态 $A$ 走到状态 $B$，得到奖励 $2$。当前表中 $V(A)=1.0$、$V(B)=3.0$；取 $\gamma=0.9$、$\alpha=0.2$。

先算目标和误差：

$$
\begin{aligned}
\text{TD target}&=2+0.9\times3.0=4.7\\
\delta&=4.7-1.0=3.7
\end{aligned}
$$

再更新 $A$：

$$
V(A)\leftarrow1.0+0.2\times3.7=1.74
$$

这次经验让 $A$ 的价值上升，但没有直接改成 4.7；它只朝目标走了 $20\%$。$B$ 的值也不是“真实答案”，它只是当前可用的后续估计。

In [ ]:
values = {"A": 1.0, "B": 3.0}
reward, gamma, alpha = 2.0, 0.9, 0.2
target = reward + gamma * values["B"]
td_error = target - values["A"]
values["A"] += alpha * td_error

print(f"TD 目标: {target:.1f}")
print(f"TD 误差: {td_error:.1f}")
print(f"更新后的 V(A): {values['A']:.2f}")

运行后应看到目标为 $4.7$、误差为 $3.7$、更新后的 $V(A)$ 为 $1.74$。代码只是在核对手算；关键是先能说明这三个数字各代表什么。

## 6. 到达终点时怎样处理？

终止状态后没有下一步未来回报，约定它的价值为 $0$。若这一步已经到达终点，TD 目标就是终止奖励：

$$
\text{TD target}=R_{t+1}\qquad\text{（终止转移）}
$$

例如进入终点得到 $+10$，当前状态的估计就朝 $10$ 更新。不要在终止状态后额外凭空加上 $\gamma V(S_{t+1})$；这会把不存在的未来奖励算进去。

## 7. TD 与蒙特卡洛：同在评价，信息不同

| 方法 | 更新所用目标 | 何时能更新 | 是否依赖当前价值估计 |
| --- | --- | --- | --- |
| 蒙特卡洛 | 完整回报 $G_t$ | 回合结束后 | 否 |
| TD(0) | $R_{t+1}+\gamma V(S_{t+1})$ | 每走一步后 | 是 |

两者都想逼近同一个策略的价值；区别在于数据的使用方式。蒙特卡洛等待更完整的信息，TD 立刻用不完整但及时的信息更新。TD 的目标会随价值表变化，所以它不像一次普通监督学习那样有固定标签。

## 8. 本课小结

- $V^{\pi}(s)$ 是固定策略 $\pi$ 下，从 $s$ 出发的期望折扣回报。
- TD 用已发生的一步奖励和下一状态的当前估计构造目标。
- TD 误差告诉我们应上调还是下调当前状态的价值。
- 终止转移没有后续价值项。
- 本课只做策略评估；下一步才讨论怎样借助动作价值来改进策略。

## 9. 自检

1. 为什么 $V^{\pi}(s)$ 要在策略 $\pi$ 固定时讨论？
2. 给定 $V(C)=5$、从 $D$ 到 $C$ 的奖励为 $-1$、$\gamma=0.8$，TD 目标是多少？
3. 如果 $V(D)=4$、$\alpha=0.5$，上题的一步 TD 更新后 $V(D)$ 是多少？
4. TD 为什么可以在未到终点时更新？它因此付出了什么代价？
5. 终止转移的目标为何不需要下一状态价值？

建议先手算第 2、3 题：目标是 $3$，更新后 $V(D)=3.5$。如果能解释每一步，就已经掌握了 TD(0) 的核心。